# Flu Season Forecasting — Exploration

This notebook walks through the CDC FluView ILI series, decomposes it into trend/seasonal/residual components, and compares SARIMA vs Prophet forecasts.

**What you'll learn**
1. How to load weekly surveillance data and align it to a regular index
2. Why we use STL decomposition for seasonal data
3. How SARIMA's `(p,d,q)(P,D,Q)_s` orders map to the ILI signal
4. How to evaluate forecasts honestly with walk-forward CV


In [ ]:
import sys; sys.path.append('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from src.data_fetcher import load_or_fetch
from src.decomposition import decompose
from src.models.arima_model import fit_sarima, forecast_sarima, SARIMAConfig
from src.models.prophet_model import fit_prophet, forecast_prophet
from src.evaluation import walk_forward_eval

## 1. Load the data

ILINet reports the % of outpatient visits flagged as influenza-like illness. We use weighted ILI (`wili`), which weights states by their population so it's representative of the US.


In [ ]:
df = load_or_fetch(Path('../data/fluview_national.csv'))
series = df.set_index('date')['wili'].asfreq('W-SAT').interpolate(limit=3).dropna()
print(f'{len(series)} weekly observations from {series.index[0].date()} to {series.index[-1].date()}')
series.plot(figsize=(13,4), title='CDC FluView Weighted %ILI');

## 2. Decomposition

STL splits the series into trend + seasonal + residual. We use `robust=True` so outliers (e.g. the 2020 COVID disruption) don't distort the seasonal component.

In [ ]:
result = decompose(series, period=52, robust=True)
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
axes[0].plot(series); axes[0].set_title('Observed')
axes[1].plot(result.trend); axes[1].set_title('Trend')
axes[2].plot(result.seasonal); axes[2].set_title('Seasonal (yearly)')
axes[3].plot(result.resid); axes[3].set_title('Residual'); axes[3].axhline(0, color='k', lw=.5)
plt.tight_layout();

**What to look for**
- Trend rises over the decade; flatlined or volatile around 2020–2022 (COVID disruption to flu)
- Seasonal component peaks in winter weeks, troughs in summer
- Residuals are mostly noise but spike at unusual seasons (2017–18 H3N2, 2022–23 early surge)

## 3. SARIMA forecast

Seasonal ARIMA: `(p,d,q)(P,D,Q)_s` where `s=52` for weekly data.
- `d=1, D=1`: one regular difference + one seasonal difference removes both trend and yearly seasonality
- `p=2, q=1`: short-range autoregression + MA on residuals
- `P=0, Q=1`: seasonal MA term to model shocks that recur yearly

In [ ]:
fit = fit_sarima(series, SARIMAConfig((2,1,1),(0,1,1,52)))
fc = forecast_sarima(fit, horizon=12)
fig, ax = plt.subplots(figsize=(13,5))
series.iloc[-156:].plot(ax=ax, label='actual', color='black')
fc['forecast'].plot(ax=ax, label='SARIMA forecast', color='crimson', lw=2)
ax.fill_between(fc.index, fc['lower'], fc['upper'], color='crimson', alpha=0.2)
ax.legend(); ax.set_title('SARIMA — 12-week forecast');

## 4. Prophet forecast

Prophet is an additive model: `y(t) = trend + seasonality + holidays + noise`. Easier to tune than SARIMA, but doesn't always beat it on clean univariate series.

In [ ]:
pm = fit_prophet(series)
pf = forecast_prophet(pm, horizon=12)
fig, ax = plt.subplots(figsize=(13,5))
series.iloc[-156:].plot(ax=ax, label='actual', color='black')
pf['forecast'].plot(ax=ax, label='Prophet forecast', color='steelblue', lw=2)
ax.fill_between(pf.index, pf['lower'], pf['upper'], color='steelblue', alpha=0.2)
ax.legend(); ax.set_title('Prophet — 12-week forecast');

## 5. Honest evaluation: walk-forward CV

Random K-fold would leak future data into training. Walk-forward CV mirrors how you'd use the model in practice: train on history, predict the next 12 weeks, roll forward, repeat.

In [ ]:
sarima_eval = walk_forward_eval(
    series, fit_fn=lambda s: fit_sarima(s, SARIMAConfig((2,1,1),(0,1,1,52))),
    forecast_fn=lambda f, h: forecast_sarima(f, h),
    horizon=12, n_folds=3, model_name='SARIMA')
prophet_eval = walk_forward_eval(
    series, fit_fn=fit_prophet, forecast_fn=forecast_prophet,
    horizon=12, n_folds=3, model_name='Prophet')
pd.DataFrame([sarima_eval.as_dict(), prophet_eval.as_dict()])

## Next steps
- Try regional or state-level forecasts (`region='hhs1'..'hhs10'`)
- Add weather or Google Trends as exogenous regressors (Prophet `add_regressor`)
- Compare against the CDC FluSight challenge baselines
- Replace point forecasts with quantile forecasts (CDC's preferred format)